In [90]:
import pandas as pd
english_data = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/English_chunks - English_chunks.csv")
english_data


,chunk,file_path,file_name,relative_path,chunk_number,chunk_id
0,"Potato J 51 (2): July-December, 2024 185Potato...",/Users/ganeshnallagachu/Desktop/world_bank/jou...,6.pdf,./6.pdf,0,6.pdf_chunk_0
1,"However, the cost of gene transfer using parti...",/Users/ganeshnallagachu/Desktop/world_bank/jou...,6.pdf,./6.pdf,1,6.pdf_chunk_1
2,"Use of 50 mg L-1 Kanamycin, 250 mg L-1 Carbeni...",/Users/ganeshnallagachu/Desktop/world_bank/jou...,6.pdf,./6.pdf,2,6.pdf_chunk_2
3,This protocol would be helpful to further stud...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,6.pdf,./6.pdf,3,6.pdf_chunk_3
4,The explants were then blotted on autoclaved a...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,6.pdf,./6.pdf,4,6.pdf_chunk_4
...,...,...,...,...,...,...
12570,(2020) reported aqueous chamber depth of 0.44 ...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,3_149686 SC.pdf,./3_149686 SC.pdf,3,3_149686 SC.pdf_chunk_3
12571,1). Vitreous chamber depth varied from 0.91 to...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,3_149686 SC.pdf,./3_149686 SC.pdf,4,3_149686 SC.pdf_chunk_4
12572,Indian Journal of Animal Sciences 94 (10) 846 ...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,3_149686 SC.pdf,./3_149686 SC.pdf,5,3_149686 SC.pdf_chunk_5
12573,Trans-corneal and trans-palpebral ultrasound s...,/Users/ganeshnallagachu/Desktop/world_bank/jou...,3_149686 SC.pdf,./3_149686 SC.pdf,6,3_149686 SC.pdf_chunk_6


In [91]:
chunk_data = pd.read_csv("/Users/ganeshnallagachu/Desktop/world_bank/chunk_data/hindi_chunks - hindi_chunks.csv")

In [92]:
chunk_data

,chunk,file_path,file_name,relative_path,chunk_number,chunk_id
0,"<!-- PageHeader=""मूल्य : ₹ 30"" --> <!-- PageHe...",/Users/ganeshnallagachu/Desktop/world_bank/dow...,खत मई 2025.txt,./खत मई 2025.txt,0,खत_मई_2025_chunk_0
1,"लेखों में व्यक्त विचारों, जानकारियों, आंकड़ों ...",/Users/ganeshnallagachu/Desktop/world_bank/dow...,खत मई 2025.txt,./खत मई 2025.txt,1,खत_मई_2025_chunk_1
2,राठौर और अंजली पटेल ## नई किस्में II भाकृअनुप ...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,खत मई 2025.txt,./खत मई 2025.txt,2,खत_मई_2025_chunk_2
3,संसाधन दक्षता और उत्पादकता बढ़ाने के लिए आधुनि...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,खत मई 2025.txt,./खत मई 2025.txt,3,खत_मई_2025_chunk_3
4,11 स जावटी मछली पालन जलीय कृषि में एक बढ़ता हु...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,खत मई 2025.txt,./खत मई 2025.txt,4,खत_मई_2025_chunk_4
...,...,...,...,...,...,...
3365,क्षेत्रों में वसंत के आगमन के साथ ही पौधरोपण क...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,फल-फल जनवर फरवर 2022.txt,./फल-फल जनवर फरवर 2022.txt,111,फल-फल_जनवर_फरवर_2022_chunk_111
3366,की खुदाई का कार्य वास्तविक वृक्षारोपण से कम से...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,फल-फल जनवर फरवर 2022.txt,./फल-फल जनवर फरवर 2022.txt,112,फल-फल_जनवर_फरवर_2022_chunk_112
3367,द्वारा भेजने की व्यवस्था करें। इस प्रकार आपको ...,/Users/ganeshnallagachu/Desktop/world_bank/dow...,फल-फल जनवर फरवर 2022.txt,./फल-फल जनवर फरवर 2022.txt,113,फल-फल_जनवर_फरवर_2022_chunk_113
3368,"<!-- PageHeader=""सामयिक"" --> ## एक ही पौधे पर ...",/Users/ganeshnallagachu/Desktop/world_bank/dow...,फल-फल जनवर फरवर 2022.txt,./फल-फल जनवर फरवर 2022.txt,114,फल-फल_जनवर_फरवर_2022_chunk_114


Create Embeddings

In [64]:
import numpy as np
import faiss
from openai import OpenAI
from tqdm import tqdm
import os
from dotenv import load_dotenv
import pickle
import time

# Load environment variables
load_dotenv()

# Initialize OpenAI client
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

client = OpenAI(api_key=openai_api_key)

print("OpenAI client initialized successfully")

OpenAI client initialized successfully


create embeddings

In [96]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Thread-safe counter for progress tracking
class ProgressCounter:
    def __init__(self, total):
        self.total = total
        self.completed = 0
        self.lock = threading.Lock()
    
    def increment(self):
        with self.lock:
            self.completed += 1
            return self.completed

def process_single_chunk(args):
    """
    Process a single chunk for embedding creation
    
    Args:
        args: tuple of (index, text, model)
    
    Returns:
        tuple: (index, embedding or None, error_message or None)
    """
    index, text, model = args
    
    # Handle NaN values
    if pd.isna(text) or text is None:
        return (index, None, "NaN value")
    
    # Convert to string
    text = str(text)
    
    # Truncate if too long (8000 tokens * 3 chars = 24000 chars for English)
    if len(text) > 24000:
        text = text[:24000].rstrip()
        # Try to break at word boundary
        last_space = text.rfind(' ')
        if last_space > 24000 * 0.9:
            text = text[:last_space]
        text = text + "..."
    
    try:
        response = client.embeddings.create(
            input=text,  # Single text, not a batch
            model=model
        )
        return (index, response.data[0].embedding, None)
        
    except Exception as e:
        return (index, None, str(e))

def get_embeddings_parallel(texts, model="text-embedding-3-small", max_workers=10):
    """
    Get embeddings for texts using parallel processing
    
    Args:
        texts: List of text chunks
        model: OpenAI embedding model
        max_workers: Number of parallel workers
    
    Returns:
        list: List of embeddings (None for failed ones)
    """
    embeddings = [None] * len(texts)  # Pre-allocate list
    errors = []
    
    # Create progress counter
    progress_counter = ProgressCounter(len(texts))
    
    # Prepare arguments for parallel processing
    args_list = [(i, text, model) for i, text in enumerate(texts)]
    
    print(f"Processing {len(texts)} chunks with {max_workers} parallel workers...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_index = {executor.submit(process_single_chunk, args): args[0] for args in args_list}
        
        # Process completed tasks
        for future in as_completed(future_to_index):
            try:
                index, embedding, error = future.result()
                embeddings[index] = embedding
                
                if error:
                    errors.append((index, error))
                
                # Update progress
                completed = progress_counter.increment()
                if completed % 100 == 0 or completed == len(texts):
                    print(f"Progress: {completed}/{len(texts)} chunks processed ({completed/len(texts)*100:.1f}%)")
                
            except Exception as e:
                index = future_to_index[future]
                errors.append((index, str(e)))
                print(f"Unexpected error processing chunk {index}: {e}")
    
    # Print summary of errors
    if errors:
        print(f"\nErrors encountered ({len(errors)} chunks):")
        for index, error in errors[:10]:  # Show first 10 errors
            print(f"  Chunk {index}: {error}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more errors")
    
    return embeddings

# Use parallel processing
print("Creating embeddings with parallel processing...")
chunk_texts = chunk_data['chunk'].tolist()

# Adjust max_workers based on your system and API rate limits
# Start with 10, increase if your API can handle more concurrent requests
max_workers = 10

embeddings = get_embeddings_parallel(chunk_texts, max_workers=max_workers)

# Filter out None embeddings and corresponding chunks
valid_indices = [i for i, emb in enumerate(embeddings) if emb is not None]
valid_embeddings = [emb for emb in embeddings if emb is not None]
valid_chunk_data = chunk_data.iloc[valid_indices].reset_index(drop=True)

print(f"\nFinal Results:")
print(f"Created {len(valid_embeddings)} valid embeddings out of {len(chunk_texts)} chunks")
print(f"Success rate: {len(valid_embeddings)/len(chunk_texts)*100:.1f}%")
print(f"Embedding dimension: {len(valid_embeddings[0]) if valid_embeddings else 0}")

Creating embeddings with parallel processing...
Processing 3370 chunks with 10 parallel workers...
Progress: 100/3370 chunks processed (3.0%)
Progress: 200/3370 chunks processed (5.9%)
Progress: 300/3370 chunks processed (8.9%)
Progress: 400/3370 chunks processed (11.9%)
Progress: 500/3370 chunks processed (14.8%)
Progress: 600/3370 chunks processed (17.8%)
Progress: 700/3370 chunks processed (20.8%)
Progress: 800/3370 chunks processed (23.7%)
Progress: 900/3370 chunks processed (26.7%)
Progress: 1000/3370 chunks processed (29.7%)
Progress: 1100/3370 chunks processed (32.6%)
Progress: 1200/3370 chunks processed (35.6%)
Progress: 1300/3370 chunks processed (38.6%)
Progress: 1400/3370 chunks processed (41.5%)
Progress: 1500/3370 chunks processed (44.5%)
Progress: 1600/3370 chunks processed (47.5%)
Progress: 1700/3370 chunks processed (50.4%)
Progress: 1800/3370 chunks processed (53.4%)
Progress: 1900/3370 chunks processed (56.4%)
Progress: 2000/3370 chunks processed (59.3%)
Progress: 210

Build FAISS Index

In [97]:
# Convert embeddings to numpy array
embeddings_array = np.array(valid_embeddings, dtype=np.float32)
print(f"Embeddings array shape: {embeddings_array.shape}")

# Normalize embeddings for cosine similarity (optional but recommended)
faiss.normalize_L2(embeddings_array)
print("Embeddings normalized for cosine similarity")

# Create FAISS index
dimension = embeddings_array.shape[1]
print(f"Creating FAISS index with dimension: {dimension}")

# Use IndexFlatIP for inner product (cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(dimension)

# Add vectors to the index
index.add(embeddings_array)
print(f"FAISS index created with {index.ntotal} vectors")

# Save the index and metadata
faiss.write_index(index, "chunk_embeddings_hindi.index")
valid_chunk_data.to_csv("chunk_embeddings_metadata_hindi.csv", index=False)
print("FAISS index and metadata saved")

Embeddings array shape: (3370, 1536)
Embeddings normalized for cosine similarity
Creating FAISS index with dimension: 1536
FAISS index created with 3370 vectors
FAISS index and metadata saved


Create FAISS Retriever Class

In [98]:
class FAISSChunkRetriever:
    def __init__(self, index_path="/Users/ganeshnallagachu/Desktop/world_bank/chunk_embeddings_hindi.index", metadata_path="/Users/ganeshnallagachu/Desktop/world_bank/chunk_embeddings_metadata_hindi.csv"):
        """Initialize the FAISS-based chunk retriever"""
        self.index = faiss.read_index(index_path)
        self.metadata = pd.read_csv(metadata_path)
        self.client = OpenAI(api_key=openai_api_key)
        
        print(f"Loaded FAISS index with {self.index.ntotal} vectors")
        print(f"Loaded metadata for {len(self.metadata)} chunks")
    
    def get_query_embedding(self, query, model="text-embedding-3-small"):
        """Get embedding for a query"""
        try:
            response = self.client.embeddings.create(
                input=query,
                model=model
            )
            embedding = response.data[0].embedding
            return np.array([embedding], dtype=np.float32)
        except Exception as e:
            print(f"Error getting query embedding: {e}")
            return None
    
    def search(self, query, k=5):
        """
        Search for the most similar chunks to a query
        
        Args:
            query (str): The search query
            k (int): Number of nearest neighbors to return (k+1 for diversity)
        
        Returns:
            list: List of dictionaries containing chunk information and similarity scores
        """
        # Get query embedding
        query_embedding = self.get_query_embedding(query)
        if query_embedding is None:
            return []
        
        # Normalize query embedding
        faiss.normalize_L2(query_embedding)
        
        # Search in FAISS index
        scores, indices = self.index.search(query_embedding, k)
        
        # Format results
        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx != -1:  # FAISS returns -1 for invalid indices
                chunk_info = self.metadata.iloc[idx].to_dict()
                results.append({
                    'rank': i + 1,
                    'chunk_id': chunk_info['chunk_id'],
                    'chunk_text': chunk_info['chunk'],
                    'file_name': chunk_info['file_name'],
                    'chunk_number': chunk_info['chunk_number'],
                    'similarity_score': float(score),
                    'file_path': chunk_info['file_path']
                })
        
        return results
    
    def search_with_diversity(self, query, k=5, diversity_threshold=0.8):
        """
        Search for diverse chunks by filtering out very similar results
        
        Args:
            query (str): The search query
            k (int): Number of diverse results to return
            diversity_threshold (float): Minimum similarity threshold for diversity
        
        Returns:
            list: List of diverse chunk results
        """
        # Get more candidates than needed
        candidates = self.search(query, k * 3)
        
        if not candidates:
            return []
        
        # Filter for diversity
        diverse_results = [candidates[0]]  # Always include the best match
        
        for candidate in candidates[1:]:
            # Check if this candidate is diverse enough from already selected results
            is_diverse = True
            for selected in diverse_results:
                # Simple heuristic: if they're from the same file and close chunk numbers, they might be similar
                if (candidate['file_name'] == selected['file_name'] and 
                    abs(candidate['chunk_number'] - selected['chunk_number']) <= 2):
                    is_diverse = False
                    break
            
            if is_diverse and len(diverse_results) < k:
                diverse_results.append(candidate)
            
            if len(diverse_results) >= k:
                break
        
        return diverse_results

# Initialize the retriever
retriever = FAISSChunkRetriever()
print("FAISS chunk retriever initialized successfully")

Loaded FAISS index with 3370 vectors
Loaded metadata for 3370 chunks
FAISS chunk retriever initialized successfully


Test the Search Functionality

In [99]:
# Test the FAISS search functionality
test_queries = [
    "However, the cost of gene transfer using particle bombardment is high, and low regeneration is often observed. Apart from this, direct gene transfer methods lead to integration of high copy number of genes resulting in gene suppression and silencing. On the contrary, Agrobacterium-mediated transformation is an efficient protocol for single gene transfer. Single gene expression is often stable to subsequent generations (Chakravarty et al., 2007). An efficient plant regeneration system is a prerequisite for developing an Agrobacterium mediated genetic transformation protocol. Successful genetic transformation and regeneration in potato is highly genotype specific. Both gene transfer and subsequent regeneration of plants is affected by several factors. These include the genotype of the plant, type of explant used, preculture time, co-cultivation time, antibiotics used for suppression of Agrobacterium overgrowth and the composition of the callus induction and AN EFFICIENT AGROBACTERIUM-MEDIATED TRANSFORMATION METHOD FOR POTATO C.V. KUFRI CHANDRAMUKHI Anupama Singh1,2, Ankita Sharma1, Hemant B. Kardille2, Vinay Bhardwaj2, Som Dutt2 and Brajesh Singh2 ABSTRACT: An optimized regeneration and Agrobacterium-mediated transformation protocol based on internode explants was developed in potato cultivar Kufri Chandramukhi. Potato internodes were transformed by cocultivation with A. tumefaciens strain EHA 105 harboring vector pRI101. MS medium with IAA 0.042mg L-1 GA3 3.0 mg L-1 Zeatin 3.0 mg L-1 showed the maximum percentage of callus formation i.e. 76 with average number of shoots per explants was 7.00. This medium showed mimimal number of days for callus initiation as compared to other medium compositions The best combination for shoot regeneration was a medium of Murashige Skoog salts with 0.042 mg L-1 IAA, 3.0 mg L-1 GA3, 3.0 mg L-1 Zeatin and 0.008 mg L-1 NAA."
]

print("Testing FAISS search with sample queries:")
print("=" * 50)

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 30)
    
    # Regular search
    results = retriever.search(query, k=5)
    
    for result in results:
        print(f"Rank {result['rank']}: {result['chunk_id']} (Score: {result['similarity_score']:.3f})")
        print(f"File: {result['file_name']}, Chunk: {result['chunk_number']}")
        print(f"Text preview: {result['chunk_text'][:100]}...")
        print()
    
    print("=" * 50)

Testing FAISS search with sample queries:

Query: However, the cost of gene transfer using particle bombardment is high, and low regeneration is often observed. Apart from this, direct gene transfer methods lead to integration of high copy number of genes resulting in gene suppression and silencing. On the contrary, Agrobacterium-mediated transformation is an efficient protocol for single gene transfer. Single gene expression is often stable to subsequent generations (Chakravarty et al., 2007). An efficient plant regeneration system is a prerequisite for developing an Agrobacterium mediated genetic transformation protocol. Successful genetic transformation and regeneration in potato is highly genotype specific. Both gene transfer and subsequent regeneration of plants is affected by several factors. These include the genotype of the plant, type of explant used, preculture time, co-cultivation time, antibiotics used for suppression of Agrobacterium overgrowth and the composition of the c

Retriving top k chunks

In [ ]:
# Function to get the best combination of chunks for a query
import json
def get_best_chunk_combination(query, k=5, use_diversity=True):
    """
    Get the best combination of chunks for a given query
    
    Args:
        query (str): The search query
        k (int): Number of chunks to return
        use_diversity (bool): Whether to use diverse search
    
    Returns:
        dict: Dictionary containing the best chunks and metadata
    """
    if use_diversity:
        results = retriever.search_with_diversity(query, k=k)
    else:
        results = retriever.search(query, k=k)
    
    if not results:
        return {
            'query': query,
            'chunks': [],
            'total_chunks': 0,
            'average_score': 0.0,
            'message': 'No relevant chunks found'
        }
    
    # Calculate statistics
    scores = [r['similarity_score'] for r in results]
    avg_score = np.mean(scores)
    
    # Group chunks by file for better organization
    chunks_by_file = {}
    for result in results:
        file_name = result['file_name']
        if file_name not in chunks_by_file:
            chunks_by_file[file_name] = []
        chunks_by_file[file_name].append(result)
    
    return {
        'query': query,
        'chunks': results,
        'chunks_by_file': chunks_by_file,
        'total_chunks': len(results),
        'average_score': avg_score,
        'score_range': (min(scores), max(scores)),
        'files_covered': len(chunks_by_file),
        'message': f'Found {len(results)} relevant chunks from {len(chunks_by_file)} files'
    }


# Test the function
# test_queries = [
#     "potato cultivation techniques",
#     "genetic modification in agriculture",
#     "crop disease management"
# ]

output_dir = "search_results"
os.makedirs(output_dir, exist_ok=True)
appendresult = []
test_queries = chunk_data['chunk'].tolist()

# Initialize the JSON file with proper structure
json_filepath = os.path.join(output_dir, "Combination_results.json")
initial_data = {
    "total_queries": 0,
    "results": []
}

# Create the initial JSON file
with open(json_filepath, "w", encoding='utf-8') as f:
    json.dump(initial_data, f, indent=2, ensure_ascii=False)

for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}/{len(test_queries)}: {query[:100]}...")
    print("=" * 50)
    
    result = get_best_chunk_combination(query, k=5, use_diversity=True)
    
    # Add metadata to result
    result['query_index'] = i
    
    appendresult.append(result)

    # Load existing data, append new result, and save back
    try:
        with open(json_filepath, "r", encoding='utf-8') as f:
            existing_data = json.load(f)
    except (json.JSONDecodeError, FileNotFoundError):
        existing_data = {
            "total_queries": 0,
            "results": []
        }
    
    # Append the new result
    existing_data["results"].append(result)
    existing_data["total_queries"] = len(existing_data["results"])
    
    # Save the updated data
    with open(json_filepath, "w", encoding='utf-8') as f:
        json.dump(existing_data, f, indent=2, ensure_ascii=False)
    
    print(f"Status: {result['message']}")
    print(f"Total chunks: {result['total_chunks']}")
    print(f"Files covered: {result['files_covered']}")
    print(f"Average score: {result['average_score']:.3f}")
    print(f"Score range: {result['score_range'][0]:.3f} - {result['score_range'][1]:.3f}")
    
    print("\nTop chunks:")
    for j, chunk in enumerate(result['chunks'][:3], 1):
        print(f"{j}. {chunk['chunk_id']} (Score: {chunk['similarity_score']:.3f})")
        print(f"   File: {chunk['file_name']}")
        print(f"   Preview: {chunk['chunk_text'][:80]}...")
    
    print("\nChunks by file:")
    for file_name, chunks in result['chunks_by_file'].items():
        print(f"  {file_name}: {len(chunks)} chunks")
    
    print("=" * 50)
    
    # Progress indicator
    if i % 100 == 0:
        print(f"\n*** Progress: {i}/{len(test_queries)} queries processed ***")

print(f"\nAll results saved to: {json_filepath}")
print(f"Total queries processed: {len(appendresult)}")

Pararallel - Retriving top k chunks

In [100]:
import json
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from datetime import datetime

def get_best_chunk_combination(query, k=5, use_diversity=True):
    """
    Get the best combination of chunks for a given query
    
    Args:
        query (str): The search query
        k (int): Number of chunks to return
        use_diversity (bool): Whether to use diverse search
    
    Returns:
        dict: Dictionary containing the best chunks and metadata
    """
    if use_diversity:
        results = retriever.search_with_diversity(query, k=k)
    else:
        results = retriever.search(query, k=k)
    
    if not results:
        return {
            'query': query,
            'chunks': [],
            'total_chunks': 0,
            'average_score': 0.0,
            'message': 'No relevant chunks found'
        }
    
    # Calculate statistics
    scores = [r['similarity_score'] for r in results]
    avg_score = np.mean(scores)
    
    # Group chunks by file for better organization
    chunks_by_file = {}
    for result in results:
        file_name = result['file_name']
        if file_name not in chunks_by_file:
            chunks_by_file[file_name] = []
        chunks_by_file[file_name].append(result)
    
    return {
        'query': query,
        'chunks': results,
        'chunks_by_file': chunks_by_file,
        'total_chunks': len(results),
        'average_score': avg_score,
        'score_range': (min(scores), max(scores)),
        'files_covered': len(chunks_by_file),
        'message': f'Found {len(results)} relevant chunks from {len(chunks_by_file)} files'
    }

def process_single_query(query_data):
    """Process a single query and return the result with index"""
    query, index = query_data
    result = get_best_chunk_combination(query, k=5, use_diversity=True)
    return result, index

# Main execution
output_dir = "search_results"
os.makedirs(output_dir, exist_ok=True)

test_queries = chunk_data['chunk'].tolist()
test_queries = test_queries
total_queries = len(test_queries)

json_filepath = os.path.join(output_dir, "Combination_all_results_hindi.json")

# Prepare query data with indices
query_data = [(query, i) for i, query in enumerate(test_queries)]

# Configure parallelization
max_workers = min(32, len(test_queries))

print(f"Starting parallel processing of {total_queries} queries with {max_workers} workers...")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Store results with their original indices to maintain order
indexed_results = {}
completed_count = 0

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_query = {executor.submit(process_single_query, qd): qd for qd in query_data}
    
    # Process completed tasks
    for future in as_completed(future_to_query):
        try:
            result, index = future.result()
            indexed_results[index] = result
            completed_count += 1
            
            # Print progress
            if completed_count <= 5 or completed_count % 100 == 0:
                query = query_data[index][0]
                print(f"\nCompleted {completed_count}/{total_queries}: {query[:100]}...")
                print(f"Status: {result['message']}")
                print(f"Total chunks: {result['total_chunks']}, Files: {result['files_covered']}")
                print(f"Average score: {result['average_score']:.3f}")
                
        except Exception as e:
            query_info = future_to_query[future]
            print(f"Error processing query {query_info[1]}: {str(e)}")

# Sort results by original order and create final structure
sorted_results = [indexed_results[i] for i in sorted(indexed_results.keys())]

# Create the exact output format you specified
final_output = {
    "total_queries": len(sorted_results),
    "results": sorted_results
}

# Save the final results in the exact format
with open(json_filepath, "w", encoding='utf-8') as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{'='*60}")
print(f"Parallel processing completed!")
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total queries processed: {len(sorted_results)}")
print(f"Results saved to: {json_filepath}")

# Final summary statistics
if sorted_results:
    total_chunks = sum(r['total_chunks'] for r in sorted_results)
    avg_score = np.mean([r['average_score'] for r in sorted_results if r['average_score'] > 0])
    print(f"Total chunks found: {total_chunks}")
    print(f"Overall average score: {avg_score:.3f}")

# Created/Modified files during execution:
# print("Combination_results.json")

Starting parallel processing of 3370 queries with 32 workers...
Started at: 2025-07-16 12:27:01

Completed 1/3370: <!-- PageHeader="नई पहल" --> ## गंगा नदी की कार्प मछलियों का पुनरुद्धार दीपेन्द्र सिंह, अभिलाष ओडेयर...
Status: Found 5 relevant chunks from 2 files
Total chunks: 5, Files: 2
Average score: 0.654

Completed 2/3370: · दाहिने हाथ के अंगूठे और तर्जनी के साथ नर से वीर्य का दोहन करना। · चिड़ियों के पंख की मदद से अंडे औ...
Status: Found 5 relevant chunks from 3 files
Total chunks: 5, Files: 3
Average score: 0.706

Completed 3/3370: मवेशियों के गोबर की मात्रा को एक तिहाई से आधे तक कम किया जा सकता है। इसके अलावा, जैविक खाद में क्रमश...
Status: Found 5 relevant chunks from 4 files
Total chunks: 5, Files: 4
Average score: 0.717

Completed 4/3370: उच्च-स्तरीय जीवन शैली के लिए ग्राहक की पसंद, पैसे में वृद्धि और घरों में सजावटी मछली रखने के मनोवैज्...
Status: Found 5 relevant chunks from 3 files
Total chunks: 5, Files: 3
Average score: 0.694

Completed 5/3370: खाद के साथ-साथ बीज उत्पाद

Filtering duplicate

In [101]:

import json
import os
from collections import defaultdict
from datetime import datetime

def load_json_file(filepath):
    """Load JSON file and return the data"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: File {filepath} not found!")
        return None
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON in {filepath}: {e}")
        return None

def get_chunk_combination_key(result):
    """
    Create a unique key for a combination based on chunk_ids
    This helps identify duplicate combinations
    """
    if 'chunks' not in result or not result['chunks']:
        return None
    
    # Sort chunk_ids to ensure consistent ordering
    chunk_ids = sorted([chunk['chunk_id'] for chunk in result['chunks']])
    return tuple(chunk_ids)

def filter_duplicate_combinations(data):
    """
    Filter out duplicate combinations from the results
    
    Args:
        data (dict): The loaded JSON data
        
    Returns:
        dict: Filtered data with duplicates removed
    """
    if not data or 'results' not in data:
        print("Error: Invalid data structure")
        return None
    
    results = data['results']
    print(f"Original number of results: {len(results)}")
    
    # Track unique combinations
    seen_combinations = set()
    unique_results = []
    duplicate_count = 0
    
    for i, result in enumerate(results):
        combination_key = get_chunk_combination_key(result)
        
        if combination_key is None:
            # Skip results with no chunks
            continue
            
        if combination_key in seen_combinations:
            duplicate_count += 1
            if duplicate_count <= 10:  # Only show first 10 duplicates to avoid spam
                print(f"Duplicate found at index {i}: {combination_key[:3]}... (first {len(combination_key)} chunks)")
        else:
            seen_combinations.add(combination_key)
            unique_results.append(result)
    
    if duplicate_count > 10:
        print(f"... and {duplicate_count - 10} more duplicates")
    
    print(f"Duplicate combinations removed: {duplicate_count}")
    print(f"Unique combinations remaining: {len(unique_results)}")
    
    # Create filtered data structure
    filtered_data = {
        "total_queries": len(unique_results),
        "results": unique_results,
        "filtering_info": {
            "original_count": len(results),
            "duplicates_removed": duplicate_count,
            "unique_remaining": len(unique_results),
            "filtered_at": datetime.now().isoformat()
        }
    }
    
    return filtered_data

def analyze_duplicates(data):
    """
    Analyze the duplicate patterns in the data
    """
    if not data or 'results' not in data:
        return
    
    results = data['results']
    combination_counts = defaultdict(int)
    
    for result in results:
        combination_key = get_chunk_combination_key(result)
        if combination_key:
            combination_counts[combination_key] += 1
    
    # Find combinations that appear multiple times
    duplicates = {k: v for k, v in combination_counts.items() if v > 1}
    
    print(f"\nDuplicate Analysis:")
    print(f"Total unique combinations: {len(combination_counts)}")
    print(f"Combinations with duplicates: {len(duplicates)}")
    
    if duplicates:
        print(f"\nTop 10 most duplicated combinations:")
        sorted_duplicates = sorted(duplicates.items(), key=lambda x: x[1], reverse=True)
        for i, (combination, count) in enumerate(sorted_duplicates[:10]):
            print(f"{i+1}. Appears {count} times: {combination[:3]}... (first {len(combination)} chunks)")

def save_filtered_data(filtered_data, output_filepath):
    """Save the filtered data to a new JSON file"""
    try:
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(filtered_data, f, indent=2, ensure_ascii=False, default=str)
        print(f"Filtered data saved to: {output_filepath}")
        return True
    except Exception as e:
        print(f"Error saving file: {e}")
        return False

# Main execution for notebook
print("=" * 60)
print("DUPLICATE COMBINATION FILTER")
print("=" * 60)

# Input file path - you can change this to your specific file
input_file = "search_results/Combination_all_results_hindi.json"

# Output file path
output_file = "search_results/Combination_all_results_filtered_hindi.json"

# Load the data
print(f"Loading data from: {input_file}")
data = load_json_file(input_file)

if data is None:
    print("Failed to load data. Please check the file path.")
else:
    # Analyze duplicates before filtering
    print("\nAnalyzing duplicates...")
    analyze_duplicates(data)
    
    # Filter duplicates
    print(f"\nFiltering duplicates...")
    filtered_data = filter_duplicate_combinations(data)
    
    if filtered_data is not None:
        # Save filtered data
        print(f"\nSaving filtered data...")
        success = save_filtered_data(filtered_data, output_file)
        
        if success:
            print(f"\n{'='*60}")
            print("FILTERING COMPLETED SUCCESSFULLY!")
            print(f"{'='*60}")
            print(f"Original file: {input_file}")
            print(f"Filtered file: {output_file}")
            print(f"Original combinations: {data['total_queries']}")
            print(f"Filtered combinations: {filtered_data['total_queries']}")
            print(f"Duplicates removed: {filtered_data['filtering_info']['duplicates_removed']}")
            reduction_percent = ((data['total_queries'] - filtered_data['total_queries']) / data['total_queries'] * 100)
            print(f"Reduction: {reduction_percent:.1f}%")
            
            # Display some sample results
            print(f"\nSample of filtered results:")
            for i, result in enumerate(filtered_data['results'][:3]):
                print(f"\nResult {i+1}:")
                print(f"  Query: {result['query'][:100]}...")
                print(f"  Chunks: {len(result['chunks'])}")
                print(f"  Average score: {result['average_score']:.3f}")
                print(f"  Files covered: {result['files_covered']}")

DUPLICATE COMBINATION FILTER
Loading data from: search_results/Combination_all_results_hindi.json

Analyzing duplicates...

Duplicate Analysis:
Total unique combinations: 3227
Combinations with duplicates: 114

Top 10 most duplicated combinations:
1. Appears 5 times: ('फल-फल_मई-जन_2025_chunk_62', 'फल-फल_मई_जन_2019_chunk_69', 'फल-फल_मई_जन_2020_chunk_93')... (first 5 chunks)
2. Appears 4 times: ('खत_अपरल_2025_chunk_98', 'खत_मई_2025_chunk_0', 'खत_मई_2025_chunk_97')... (first 5 chunks)
3. Appears 4 times: ('खत_मरच_2024_chunk_27', 'खत_मरच_2024_chunk_38', 'खत_मरच_2025_chunk_26')... (first 5 chunks)
4. Appears 4 times: ('फल-फल_जनवर_फरवर_2021_chunk_29', 'फल-फल_जनवर_फरवर_2021_chunk_70', 'फल-फल_जनवर_फरवर_2021_chunk_76')... (first 5 chunks)
5. Appears 4 times: ('फल-फल_जनवर_फरवर_2022_chunk_102', 'फल-फल_जलई-अगसत_2022_chunk_90', 'फल-फल_नवबर_-_दसबर_2021_chunk_75')... (first 5 chunks)
6. Appears 4 times: ('फल-फल_मई-जन_2025_chunk_60', 'फल-फल_मई_जन_2019_chunk_74', 'फल-फल_मई_जन_2020_chunk_98')... (first 

converting to CSV

In [102]:
# Convert Combination_all_results_filtered.json to CSV format
# Copy this code into a new cell in your notebook

import json
import pandas as pd

def convert_json_to_csv(json_filepath, csv_filepath):
    """Convert JSON results to CSV format with chunk_id and chunks columns"""
    
    # Load JSON data
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    csv_data = []
    
    for i, combination in enumerate(data['results']):
        # Extract chunk IDs and combine them
        chunk_ids = []
        chunk_texts = []
        
        for j, chunk in enumerate(combination['chunks']):
            chunk_ids.append(chunk['chunk_id'])
            chunk_texts.append(f"chunk{j+1}: {chunk['chunk_text']}")
        
        # Combine all chunk IDs
        combined_chunk_ids = " + ".join(chunk_ids)
        
        # Combine all chunk texts
        combined_chunks = " | ".join(chunk_texts)
        
        # Create row data
        row_data = {
            'chunk_id': combined_chunk_ids,
            'chunks': combined_chunks
        }
        
        csv_data.append(row_data)
    
    # Create DataFrame
    df = pd.DataFrame(csv_data)
    
    # Save to CSV
    df.to_csv(csv_filepath, index=False, encoding='utf-8')
    
    return df

# Convert the file
df = convert_json_to_csv('search_results/Combination_all_results_filtered_hindi.json', 'final_combination_results_hindi.csv')

# Display first few rows
print("CSV Conversion Complete!")
print(f"Total combinations: {len(df)}")
print("\nFirst 3 rows:")
print(df.head(3))

# Show column info
print(f"\nColumns: {list(df.columns)}")
print(f"CSV file saved as: combination_results.csv")

# Store DataFrame for further use
combination_csv_data = df

CSV Conversion Complete!
Total combinations: 3227

First 3 rows:
                                            chunk_id  \
0  खत_मई_2025_chunk_0 + खत_मई_2025_chunk_66 + फल-...   
1  खत_मई_2025_chunk_1 + खत_अपरल_2025_chunk_4 + खत...   
2  खत_मई_2025_chunk_2 + खत_मरच_2024_chunk_70 + खत...   

                                              chunks  
0  chunk1: <!-- PageHeader="मूल्य : ₹ 30" --> <!-...  
1  chunk1: लेखों में व्यक्त विचारों, जानकारियों, ...  
2  chunk1: राठौर और अंजली पटेल ## नई किस्में II भ...  

Columns: ['chunk_id', 'chunks']
CSV file saved as: combination_results.csv


LLM QA pair generation

In [28]:
# LLM Function to Generate QA Pairs from Combinations
# Copy this code into a new cell in your Jupyter notebook

import json
import os
from openai import OpenAI
from dotenv import load_dotenv
import time
from tqdm import tqdm

# Load environment variables
load_dotenv()

# Initialize OpenAI client
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

client = OpenAI(api_key=openai_api_key)

def format_chunks_for_prompt(chunks):
    """
    Format chunks for the LLM prompt
    """
    formatted_chunks = []
    for i, chunk in enumerate(chunks, 1):
        # chunk_text = chunk['chunk_text'][:500] + "..." if len(chunk['chunk_text']) > 500 else chunk['chunk_text']
        formatted_chunks.append(f"Chunk {i}: {chunk}")
    
    return "\n\n".join(formatted_chunks)

def generate_qa_pairs_for_combination(combination, max_retries=3):
    """
    Generate QA pairs for a single combination using the LLM
    
    Args:
        combination (dict): A single combination result
        max_retries (int): Maximum number of retry attempts
        
    Returns:
        dict: Combination with added QA pairs
    """
    
    # Prepare the prompt
    chunks_text = format_chunks_for_prompt(combination['chunks'])
    
    prompt = f"""You are an AI assistant specializing in creating complex, multi-source Question-Answer pairs that require synthesis and analysis across multiple text chunks. Your goal is to generate questions that cannot be answered using any single chunk alone.

User Persona
Target User: A farmer with 8th-10th grade education who may:

Have limited English proficiency with spelling mistakes
Use colloquial language and regional expressions
Ask vague or imprecise questions initially
Need practical, actionable agricultural advice

Input Context
You will receive multiple related text chunks: 

{chunks_text}

Task Requirements
Generate as many Multi-Chunk question answer Pairs as possible but strictly avoid being repetitive. If no multi-chunk question answer pairs are possible, not required to force.
Each question must meet these criteria:

Multi-chunk dependency: Answer requires information from at least 2 chunks
Synthesis requirement: Cannot be answered by simply listing facts from different chunks
Agricultural relevance: Focuses on farming, crops, livestock, or soil management
User-appropriate language: Simple, clear English reflecting the farmer persona

Question Categories (Priority Order)
1. Problem-Solution Integration

Link problems mentioned in one chunk with solutions from another
Example: "My crops showing yellow leaves, what causing this and how fix it?"

2. Comparative Decision Making

Compare methods, treatments, or approaches across chunks
Example: "Which better for my soil - organic manure or chemical fertilizer and why?"

3. Cause-Effect Relationships

Connect causes from one chunk with effects described in another
Example: "If I not rotate crops like book say, what happen to my field?"

4. Process Integration

Link sequential steps or stages mentioned across chunks
Example: "From planting to harvest, what most important things for good crop?"

5. Conditional Application

Apply principles from one chunk to scenarios in another
Example: "My area get less rain, can I still use method they talk about?"

Quality Standards
For Questions:
Use simple, conversational language with occasional grammatical imperfections
Include practical context ("my field", "my crops", "in my area")
Focus on actionable information
Avoid technical jargon unless commonly known
Length: 10-25 words

For Answers:
Synthesize information from multiple chunks explicitly
Provide practical, implementable advice
Use simple language but remain accurate
Include specific details from the chunks
Length: 50-150 words
Structure: Problem acknowledgment → Explanation → Practical solution

Output Format
**Question 1:** [Question in farmer's language]
**Answer 1:** [Synthesized answer drawing from multiple chunks]
**Chunks Used:** [List chunk numbers used]

**Question 2:** [Question in farmer's language]
**Answer 2:** [Synthesized answer drawing from multiple chunks]
**Chunks Used:** [List chunk numbers used]

Examples (Based on Potato Improvement Context)
GOOD Multi-Chunk Question:
Question: "Scientists say potato need new method for better crop, but gun method and bacteria method both costly - which one farmer like me should choose and why?"
Answer: Based on the research, bacteria method (Agrobacterium) is better choice for farmers. While gun method works, it is very expensive and often plants don't grow back properly. Bacteria method is more efficient, costs less, and gives better results - scientists got 76% success rate with 7 shoots per plant using this method. For potato improvement, bacteria method is more practical option.
Chunks Used: 1, 2

BAD Single-Chunk Question:
Question: "What are methods for potato improvement?"
Why Bad: Can be answered from Chunk 1 alone, no synthesis required

Validation Checklist
Before finalizing each QA pair, verify:

 Question requires information from 2+ chunks
 Answer synthesizes rather than lists information
 Language matches farmer persona
 Content is agriculturally relevant and practical
 Answer provides actionable advice
 All claims are supported by the provided chunks

Generate QA pairs now:"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",  # You can change to gpt-4 if needed
                messages=[
                    {"role": "system", "content": "You are an expert agricultural AI assistant that creates high-quality, multi-source question-answer pairs for farmers."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=1500,
                temperature=0.7,
                top_p=0.9
            )
            
            qa_text = response.choices[0].message.content.strip()
            
            # Parse the QA pairs from the response
            qa_pairs = parse_qa_pairs(qa_text)
            
            # Add QA pairs to the combination
            combination['qa_pairs'] = qa_pairs
            combination['qa_generation_status'] = 'success'
            
            return combination
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(2)  # Wait before retrying
            else:
                print(f"Failed to generate QA pairs after {max_retries} attempts")
                combination['qa_pairs'] = []
                combination['qa_generation_status'] = 'failed'
                combination['qa_error'] = str(e)
                return combination

def parse_qa_pairs(qa_text):
    """
    Parse QA pairs from the LLM response
    """
    qa_pairs = []
    
    # Split by question markers
    parts = qa_text.split('**Question')
    
    for part in parts[1:]:  # Skip the first empty part
        try:
            # Extract question
            question_start = part.find(':**') + 3
            answer_start = part.find('**Answer')
            
            if question_start > 2 and answer_start > question_start:
                question = part[question_start:answer_start].strip()
                
                # Extract answer
                answer_part = part[answer_start:]
                answer_start_pos = answer_part.find(':**') + 3
                chunks_used_start = answer_part.find('**Chunks Used:**')
                
                if answer_start_pos > 2 and chunks_used_start > answer_start_pos:
                    answer = answer_part[answer_start_pos:chunks_used_start].strip()
                    
                    # Extract chunks used
                    chunks_used_text = answer_part[chunks_used_start + 16:].strip()
                    chunks_used = [int(x.strip()) for x in chunks_used_text.strip('[]').split(',') if x.strip().isdigit()]
                    
                    qa_pairs.append({
                        'question': question,
                        'answer': answer,
                        'chunks_used': chunks_used
                    })
        except Exception as e:
            print(f"Error parsing QA pair: {e}")
            continue
    
    return qa_pairs

def generate_qa_pairs_for_all_combinations(filtered_data, output_file, batch_size=10):
    """
    Generate QA pairs for all combinations in the filtered data
    
    Args:
        filtered_data (dict): The filtered combinations data
        output_file (str): Output file path
        batch_size (int): Number of combinations to process before saving
    """
    
    results = filtered_data['results'][1:20]
    total_combinations = len(results)
    
    print(f"Starting QA pair generation for {total_combinations} combinations...")
    print(f"Output will be saved to: {output_file}")
    
    # Initialize output data
    output_data = {
        "total_combinations": total_combinations,
        "qa_generation_info": {
            "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_size": batch_size
        },
        "results": []
    }
    
    # Process combinations in batches
    for i in tqdm(range(0, total_combinations, batch_size), desc="Generating QA pairs"):
        batch = results[i:i + batch_size]
        
        for j, combination in enumerate(batch):
            print(f"\nProcessing combination {i + j + 1}/{total_combinations}")
            
            # Generate QA pairs for this combination
            combination_with_qa = generate_qa_pairs_for_combination(combination)
            output_data['results'].append(combination_with_qa)
            
            # Add small delay to avoid rate limiting
            time.sleep(1)
        
        # Save progress after each batch
        output_data['qa_generation_info']['last_saved'] = time.strftime("%Y-%m-%d %H:%M:%S")
        output_data['qa_generation_info']['combinations_processed'] = len(output_data['results'])
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False, default=str)
        
        print(f"Saved progress: {len(output_data['results'])}/{total_combinations} combinations processed")
    
    # Final save
    output_data['qa_generation_info']['completed_at'] = time.strftime("%Y-%m-%d %H:%M:%S")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"\n{'='*60}")
    print("QA PAIR GENERATION COMPLETED!")
    print(f"{'='*60}")
    print(f"Total combinations processed: {len(output_data['results'])}")
    print(f"Output saved to: {output_file}")
    
    # Show statistics
    successful = sum(1 for r in output_data['results'] if r.get('qa_generation_status') == 'success')
    failed = sum(1 for r in output_data['results'] if r.get('qa_generation_status') == 'failed')
    total_qa_pairs = sum(len(r.get('qa_pairs', [])) for r in output_data['results'])
    
    print(f"Successful generations: {successful}")
    print(f"Failed generations: {failed}")
    print(f"Total QA pairs generated: {total_qa_pairs}")
    print(f"Average QA pairs per combination: {total_qa_pairs/len(output_data['results']):.1f}")

# Main execution
if __name__ == "__main__":
    # Load the filtered data
    input_file = "search_results/Combination_5_results_filtered.json"
    output_file = "search_results/Combination_5_results_with_qa.json"
    
    print("Loading filtered combinations...")
    with open(input_file, 'r', encoding='utf-8') as f:
        filtered_data = json.load(f)
    
    print(f"Loaded {len(filtered_data['results'])} combinations")
    
    # Generate QA pairs
    generate_qa_pairs_for_all_combinations(filtered_data, output_file, batch_size=5)

Loading filtered combinations...
Loaded 917 combinations
Starting QA pair generation for 19 combinations...
Output will be saved to: search_results/Combination_5_results_with_qa.json


Generating QA pairs:   0%|          | 0/4 [00:00<?, ?it/s]


Processing combination 1/19

Processing combination 2/19

Processing combination 3/19

Processing combination 4/19

Processing combination 5/19


Generating QA pairs:  25%|██▌       | 1/4 [01:18<03:54, 78.04s/it]

Saved progress: 5/19 combinations processed

Processing combination 6/19

Processing combination 7/19

Processing combination 8/19

Processing combination 9/19

Processing combination 10/19


...

Saved progress: 10/19 combinations processed

Processing combination 11/19

Processing combination 12/19

Processing combination 13/19

Processing combination 14/19

Processing combination 15/19


Generating QA pairs:  75%|███████▌  | 3/4 [03:32<01:09, 69.80s/it]

Saved progress: 15/19 combinations processed

Processing combination 16/19

Processing combination 17/19

Processing combination 18/19

Processing combination 19/19


Generating QA pairs: 100%|██████████| 4/4 [04:22<00:00, 65.57s/it]

Saved progress: 19/19 combinations processed

QA PAIR GENERATION COMPLETED!
Total combinations processed: 19
Output saved to: search_results/Combination_5_results_with_qa.json
Successful generations: 19
Failed generations: 0
Total QA pairs generated: 94
Average QA pairs per combination: 4.9


In [29]:
# Extract and format QA results
# Copy this code into a new cell in your notebook

import json

def extract_qa_results(filepath, num_results=3):
    """Extract and format QA results from JSON file"""
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    formatted_results = []
    
    for i, combination in enumerate(data['results']):
        # Format chunks
        chunks_text = []
        for chunk in combination['chunks']:
            chunks_text.append(chunk['chunk_text'])
        
        # Format QA pairs (without chunks_used)
        qa_pairs = []
        for qa in combination['qa_pairs']:
            qa_pairs.append({
                "question": qa['question'],
                "answer": qa['answer']
            })
        
        # Create formatted result
        result = {
            f"chunks": chunks_text,
            "qa_pairs": qa_pairs
        }
        
        formatted_results.append(result)
    
    return formatted_results

# Load and extract results
results = extract_qa_results('search_results/Combination_5_results_with_qa.json', num_results=3)
#save results
with open('search_results/llm_eval_5.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Print formatted results
for i, result in enumerate(results):
    print(f"\n{'='*50}")
    print(f"RESULT {i+1}")
    print(f"{'='*50}")
    
    # Print chunks
    print(f"{i+1}.chunks:")
    for j, chunk_text in enumerate(result["chunks"]):
        print(f"   Chunk {j+1}: {chunk_text[:200]}...")
    
    # Print QA pairs
    print(f"\nqa_pairs:")
    for k, qa in enumerate(result['qa_pairs']):
        print(f"   Q{k+1}: {qa['question']}")
        print(f"   A{k+1}: {qa['answer'][:150]}...")
        print()

# Store results for further use
extracted_results = results
print(f"Extracted {len(results)} results. Data stored in 'extracted_results' variable.")


RESULT 1
1.chunks:
   Chunk 1: However, the cost of gene transfer using particle bombardment is high, and low regeneration is often observed. Apart from this, direct gene transfer methods lead to integration of high copy number of ...
   Chunk 2: tumefaciens strain EHA 105 harboring vector pRI101. MS medium with IAA 0.042mg L-1 GA3 3.0 mg L-1 Zeatin 3.0 mg L-1 showed the maximum percentage of callus formation i.e. 76 with average number of sho...
   Chunk 3: The callus induction and regeneration reponse in potato is highly genotype specific Efficient transformation and regeneration protocol is critical for developing transgenics gene editing and function ...
   Chunk 4: Expt. Botany49: 1589-1595. Chakravarty B Wang-Pruski, G Flinn B Gustafson V and Regan S (2007) Genetic transformation in potato: Approaches and strategies. American Journal of Potato Research. 84: 301...
   Chunk 5: The sequence of the forward and reverse primers was 5 GAGGCTATTCGGCTATGAGTG 3 and 5 GCGATACCGTAAAGCACGAG

In [35]:
# Calculate token usage for QA combinations
# Copy this code into a new cell in your notebook

import json
import tiktoken
from collections import defaultdict

def count_tokens(text, model="gpt-4o-mini"):
    """Count tokens in text using tiktoken"""
    try:
        encoding = tiktoken.encoding_for_model(model)
        return len(encoding.encode(text))
    except:
        # Fallback to cl100k_base encoding
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))

def calculate_token_usage(filepath):
    """Calculate token usage for each combination"""
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    token_stats = []
    
    for i, combination in enumerate(data['results']):
        # Get chunks text
        chunks_text = []
        for chunk in combination['chunks']:
            chunks_text.append(chunk['chunk_text'])
        
        # Calculate input tokens (chunks)
        input_tokens = sum(count_tokens(chunk) for chunk in chunks_text)
        
        # Calculate output tokens (QA pairs)
        output_tokens = 0
        qa_count = 0
        
        for qa in combination['qa_pairs']:
            question_tokens = count_tokens(qa['question'])
            answer_tokens = count_tokens(qa['answer'])
            output_tokens += question_tokens + answer_tokens
            qa_count += 1
        
        # Calculate average output tokens per QA pair
        avg_output_tokens = output_tokens / qa_count if qa_count > 0 else 0
        
        # Calculate prompt tokens (your original prompt template)
        prompt_template = """You are an AI assistant specializing in creating complex, multi-source Question-Answer pairs that require synthesis and analysis across multiple text chunks. Your goal is to generate questions that cannot be answered using any single chunk alone.

User Persona
Target User: A farmer with 8th-10th grade education who may:

Have limited English proficiency with spelling mistakes
Use colloquial language and regional expressions
Ask vague or imprecise questions initially
Need practical, actionable agricultural advice

Input Context
You will receive 5 related text chunks: {chunks}

Task Requirements
Generate as many Multi-Chunk question answer Pairs as possible but strictly avoid being repetitive. If  no multi-chunk question answer pairs are possible, not required to force.
Each question must meet these criteria:

Multi-chunk dependency: Answer requires information from at least 2 chunks
Synthesis requirement: Cannot be answered by simply listing facts from different chunks
Agricultural relevance: Focuses on farming, crops, livestock, or soil management
User-appropriate language: Simple, clear English reflecting the farmer persona

Question Categories (Priority Order)
1. Problem-Solution Integration

Link problems mentioned in one chunk with solutions from another
Example: "My crops showing yellow leaves, what causing this and how fix it?"

2. Comparative Decision Making

Compare methods, treatments, or approaches across chunks
Example: "Which better for my soil - organic manure or chemical fertilizer and why?"

3. Cause-Effect Relationships

Connect causes from one chunk with effects described in another
Example: "If I not rotate crops like book say, what happen to my field?"

4. Process Integration

Link sequential steps or stages mentioned across chunks
Example: "From planting to harvest, what most important things for good crop?"

5. Conditional Application

Apply principles from one chunk to scenarios in another
Example: "My area get less rain, can I still use method they talk about?"

Quality Standards
For Questions:
Use simple, conversational language with occasional grammatical imperfections
Include practical context ("my field", "my crops", "in my area")
Focus on actionable information
Avoid technical jargon unless commonly known
Length: 10-25 words

For Answers:
Synthesize information from multiple chunks explicitly
Provide practical, implementable advice
Use simple language but remain accurate
Include specific details from the chunks
Length: 50-150 words
Structure: Problem acknowledgment → Explanation → Practical solution

Output Format
**Question 1:** [Question in farmer's language]
**Answer 1:** [Synthesized answer drawing from multiple chunks]
**Chunks Used:** [List chunk numbers used]

**Question 2:** [Question in farmer's language]
**Answer 2:** [Synthesized answer drawing from multiple chunks]
**Chunks Used:** [List chunk numbers used]

E"""
        
        # Format chunks for prompt
        formatted_chunks = []
        for j, chunk in enumerate(chunks_text):
            formatted_chunks.append(f"Chunk {j+1}: {chunk}")
        
        chunks_text_combined = "\n\n".join(formatted_chunks)
        
        prompt_tokens = count_tokens(prompt_template)
        
        # Store statistics
        stats = {
            'combination_id': i,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'avg_output_tokens_per_qa': avg_output_tokens,
            'prompt_tokens': prompt_tokens,
            # 'total_tokens': prompt_tokens + output_tokens,
            'qa_count': qa_count,
            'chunk_count': len(chunks_text)
        }
        
        token_stats.append(stats)
    
    return token_stats

# Calculate token usage
token_stats = calculate_token_usage('search_results/Combination_5_results_with_qa.json')

# Print results
print("TOKEN USAGE STATISTICS")
print("=" * 80)

for stats in token_stats:
    print(f"\nCombination {stats['combination_id'] + 1}:")
    print(f"  Input Tokens (chunks): {stats['input_tokens']:,}")
    print(f"  Output Tokens (total): {stats['output_tokens']:,}")
    print(f"  Avg Output Tokens per QA: {stats['avg_output_tokens_per_qa']:.1f}")
    print(f"  Prompt Tokens: {stats['prompt_tokens']:,}")
    # print(f"  Total Tokens: {stats['total_tokens']:,}")
    print(f"  QA Pairs: {stats['qa_count']}")
    print(f"  Chunks: {stats['chunk_count']}")

# Calculate averages across all combinations
total_combinations = len(token_stats)
avg_input_tokens = sum(s['input_tokens'] for s in token_stats) / total_combinations
avg_output_tokens = sum(s['output_tokens'] for s in token_stats) / total_combinations
avg_prompt_tokens = sum(s['prompt_tokens'] for s in token_stats) / total_combinations
# avg_total_tokens = sum(s['total_tokens'] for s in token_stats) / total_combinations
avg_qa_per_combination = sum(s['qa_count'] for s in token_stats) / total_combinations

print(f"\n{'='*80}")
print("OVERALL AVERAGES")
print(f"{'='*80}")
print(f"Average Input Tokens per Combination: {avg_input_tokens:,.1f}")
print(f"Average Output Tokens per Combination: {avg_output_tokens:,.1f}")
print(f"Average Prompt Tokens per Combination: {avg_prompt_tokens:,.1f}")
# print(f"Average Total Tokens per Combination: {avg_total_tokens:,.1f}")
print(f"Average QA Pairs per Combination: {avg_qa_per_combination:.1f}")

# Store results for further analysis
token_usage_stats = token_stats
print(f"\nToken usage statistics stored in 'token_usage_stats' variable.")

TOKEN USAGE STATISTICS

Combination 1:
  Input Tokens (chunks): 3,131
  Output Tokens (total): 629
  Avg Output Tokens per QA: 125.8
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 2:
  Input Tokens (chunks): 3,150
  Output Tokens (total): 684
  Avg Output Tokens per QA: 136.8
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 3:
  Input Tokens (chunks): 2,977
  Output Tokens (total): 563
  Avg Output Tokens per QA: 112.6
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 4:
  Input Tokens (chunks): 2,994
  Output Tokens (total): 617
  Avg Output Tokens per QA: 123.4
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 5:
  Input Tokens (chunks): 3,361
  Output Tokens (total): 575
  Avg Output Tokens per QA: 115.0
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 6:
  Input Tokens (chunks): 2,839
  Output Tokens (total): 641
  Avg Output Tokens per QA: 128.2
  Prompt Tokens: 566
  QA Pairs: 5
  Chunks: 5

Combination 7:
  Input Tokens (chu